In [33]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = (
    Path("../data/processed")
    / "Ambazari_Nagpur_Selected_Features_Dataset_2024_2025.csv"
)

data = pd.read_csv(data_path)

print("Dataset loaded successfully")
print("Shape:", data.shape)
print("Columns:", data.columns.tolist())
print("Missing values:", data.isna().sum().sum())
print("Duplicate rows:", data.duplicated().sum())

display(data.head())

Dataset loaded successfully
Shape: (17519, 46)
Columns: ['timestamp', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'temperature', 'humidity', 'wind_speed', 'weather_code', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'wind_direction_sin', 'wind_direction_cos', 'season', 'pm25_lag_1', 'pm25_lag_3', 'pm25_lag_6', 'pm25_lag_12', 'pm25_lag_24', 'pm10_lag_1', 'pm10_lag_24', 'no2_lag_1', 'no2_lag_24', 'so2_lag_1', 'co_lag_1', 'o3_lag_1', 'o3_lag_24', 'pm25_rolling_mean_3', 'pm25_rolling_mean_6', 'pm25_rolling_mean_12', 'pm25_rolling_mean_24', 'pm10_rolling_mean_3', 'pm10_rolling_mean_6', 'pm10_rolling_mean_24', 'no2_rolling_mean_3', 'o3_rolling_mean_3', 'target_pm25', 'target_pm10', 'target_no2', 'target_so2', 'target_co', 'target_o3']
Missing values: 0
Duplicate rows: 0


,timestamp,pm25,pm10,no2,so2,co,o3,temperature,humidity,wind_speed,...,pm10_rolling_mean_6,pm10_rolling_mean_24,no2_rolling_mean_3,o3_rolling_mean_3,target_pm25,target_pm10,target_no2,target_so2,target_co,target_o3
0,2024-01-02 00:00:00,110.18,201.10,51.48,8.60,1.95,2.25,22.7,43,1.583333,...,200.871667,160.033536,76.006667,2.100000,102.98,192.09,56.93,11.93,2.10,2.06
1,2024-01-02 01:00:00,102.98,192.09,56.93,11.93,2.10,2.06,21.7,46,1.416667,...,211.261667,164.318079,58.240000,1.193333,100.73,184.74,54.57,11.10,2.35,1.47
2,2024-01-02 02:00:00,100.73,184.74,54.57,11.10,2.35,1.47,20.6,50,0.861111,...,214.148333,168.227205,55.516667,1.516667,100.38,179.77,45.12,10.40,2.00,3.54
3,2024-01-02 03:00:00,100.38,179.77,45.12,10.40,2.00,3.54,16.6,70,1.861111,...,211.975000,171.830081,54.326667,1.926667,97.18,177.32,24.54,10.21,0.96,9.84
4,2024-01-02 04:00:00,97.18,177.32,24.54,10.21,0.96,9.84,15.9,76,1.750000,...,205.266667,175.225874,52.206667,2.356667,91.28,170.08,18.30,10.13,0.60,15.36


In [34]:
# Convert timestamp and sort chronologically
data["timestamp"] = pd.to_datetime(data["timestamp"])
data = data.sort_values("timestamp").reset_index(drop=True)

# Define next-hour prediction targets
targets = [
    "target_pm25",
    "target_pm10",
    "target_no2",
    "target_so2",
    "target_co",
    "target_o3"
]

# Verify all targets exist
missing_targets = [col for col in targets if col not in data.columns]

if missing_targets:
    raise ValueError(f"Missing target columns: {missing_targets}")

# Separate features and targets
X = data.drop(columns=targets + ["timestamp"])
y = data[targets]

# Display dataset details
print("Date range:", data["timestamp"].min(), "to", data["timestamp"].max())
print("Total rows:", len(data))
print("Number of features:", X.shape[1])
print("Number of targets:", y.shape[1])

print("\nTarget columns:")
print(y.columns.tolist())

print("\nFeature data types:")
display(X.dtypes.value_counts().rename_axis("Data type").to_frame("Count"))

print("\nTarget summary:")
display(y.describe().T.round(3))

Date range: 2024-01-02 00:00:00 to 2025-12-31 22:00:00
Total rows: 17519
Number of features: 39
Number of targets: 6

Target columns:
['target_pm25', 'target_pm10', 'target_no2', 'target_so2', 'target_co', 'target_o3']

Feature data types:


,Count
Data type,
float64,36
int64,2
str,1



Target summary:


,count,mean,std,min,25%,50%,75%,max
target_pm25,17519.0,46.214,27.092,0.05,24.65,42.00,63.270,487.48
target_pm10,17519.0,91.281,45.996,6.19,56.60,86.38,117.550,832.54
target_no2,17519.0,21.054,19.182,0.01,9.02,15.16,25.550,177.75
target_so2,17519.0,16.950,18.081,0.02,8.11,11.86,18.295,198.76
target_co,17519.0,0.645,0.541,0.00,0.33,0.48,0.730,7.63
target_o3,17519.0,35.942,32.438,0.00,9.86,25.97,53.260,187.86


In [ ]:
# Check timestamp continuity
time_diff = data["timestamp"].diff()

print("Duplicate timestamps:", data["timestamp"].duplicated().sum())
print("Missing or irregular hourly intervals:", (time_diff > pd.Timedelta(hours=1)).sum())
print("Intervals shorter than one hour:", (time_diff < pd.Timedelta(hours=1)).sum())
print("Largest time gap:", time_diff.max())

# Display any gaps larger than one hour
gap_indices = data.index[time_diff > pd.Timedelta(hours=1)]

if len(gap_indices) > 0:
    gap_report = pd.DataFrame({
        "Previous timestamp": data.loc[gap_indices - 1, "timestamp"].values,
        "Next timestamp": data.loc[gap_indices, "timestamp"].values,
        "Gap duration": time_diff.loc[gap_indices].values
    })
    display(gap_report.head(20))    
else:
    print("No gaps larger than one hour.")

# Inspect target alignment around a few consecutive hours
alignment_check = data[
    ["timestamp", "pm25", "target_pm25"]
].head(10)

display(alignment_check)

Duplicate timestamps: 0
Missing or irregular hourly intervals: 0
Intervals shorter than one hour: 0
Largest time gap: 0 days 01:00:00
No gaps larger than one hour.


,timestamp,pm25,target_pm25
0,2024-01-02 00:00:00,110.18,102.98
1,2024-01-02 01:00:00,102.98,100.73
2,2024-01-02 02:00:00,100.73,100.38
3,2024-01-02 03:00:00,100.38,97.18
4,2024-01-02 04:00:00,97.18,91.28
5,2024-01-02 05:00:00,91.28,85.43
6,2024-01-02 06:00:00,85.43,82.68
7,2024-01-02 07:00:00,82.68,82.17
8,2024-01-02 08:00:00,82.17,82.83
9,2024-01-02 09:00:00,82.83,87.26


In [ ]:
      # Chronological split: 70% train, 15% validation, 15% test

      n = len(data)

      train_end = int(n * 0.70)
      val_end = int(n * 0.85)

      X_train = X.iloc[:train_end].copy()
      X_val = X.iloc[train_end:val_end].copy()
      X_test = X.iloc[val_end:].copy()

      y_train = y.iloc[:train_end].copy()
      y_val = y.iloc[train_end:val_end].copy()
      y_test = y.iloc[val_end:].copy()

      print("TRAINING SET")
      print("Rows:", len(X_train))
      print("Period:", data["timestamp"].iloc[0], "to",
            data["timestamp"].iloc[train_end - 1])

      print("\nVALIDATION SET")
      print("Rows:", len(X_val))
      print("Period:", data["timestamp"].iloc[train_end], "to",
            data["timestamp"].iloc[val_end - 1])

      print("\nTEST SET")
      print("Rows:", len(X_test))
      print("Period:", data["timestamp"].iloc[val_end], "to",
            data["timestamp"].iloc[-1])

      print("\nTotal rows:", len(X_train) + len(X_val) + len(X_test))

TRAINING SET
Rows: 12263
Period: 2024-01-02 00:00:00 to 2025-05-26 22:00:00

VALIDATION SET
Rows: 2628
Period: 2025-05-26 23:00:00 to 2025-09-13 10:00:00

TEST SET
Rows: 2628
Period: 2025-09-13 11:00:00 to 2025-12-31 22:00:00

Total rows: 17519


In [63]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

baseline_results = []

for target, current_col in pollutant_columns.items():
    actual = y_val[target].to_numpy()
    predicted = X_val[current_col].to_numpy()

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)

    baseline_results.append({
        "Pollutant": target.replace("target_", "").upper(),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

baseline_results_df = pd.DataFrame(baseline_results)

display(
    baseline_results_df
    .sort_values("Pollutant")
    .round(4)
)

,Pollutant,MAE,RMSE,R2
4,CO,0.1082,0.1902,0.5092
2,NO2,2.8377,4.5569,0.6199
5,O3,3.4558,7.9942,0.7392
1,PM10,4.4909,6.4054,0.9239
0,PM25,3.3081,4.5485,0.8714
3,SO2,0.9103,2.8434,0.7895


In [38]:
# Prepare target transformations

y_train_original = y_train.copy()
y_val_original = y_val.copy()
y_test_original = y_test.copy()

y_train_log = np.log1p(y_train_original)

print("Original training target shape:", y_train_original.shape)
print("Log-transformed training target shape:", y_train_log.shape)

print("\nTraining target skewness before and after log1p:")

skewness_comparison = pd.DataFrame({
    "Original skewness": y_train_original.skew(),
    "Log1p skewness": y_train_log.skew()
})

display(skewness_comparison.round(3))

print("\nLog-transformed target summary:")
display(y_train_log.describe().T.round(3))

Original training target shape: (12263, 6)
Log-transformed training target shape: (12263, 6)

Training target skewness before and after log1p:


,Original skewness,Log1p skewness
target_pm25,1.467,-0.660
target_pm10,2.260,-0.349
target_no2,2.337,0.032
target_so2,4.039,0.539
target_co,2.824,1.523
target_o3,0.883,-0.709



Log-transformed target summary:


,count,mean,std,min,25%,50%,75%,max
target_pm25,12263.0,3.773,0.588,0.049,3.414,3.858,4.205,6.191
target_pm10,12263.0,4.521,0.445,2.328,4.240,4.560,4.828,6.726
target_no2,12263.0,2.919,0.743,0.010,2.434,2.896,3.387,5.186
target_so2,12263.0,2.675,0.677,0.020,2.226,2.616,3.064,5.297
target_co,12263.0,0.488,0.271,0.000,0.307,0.419,0.582,2.155
target_o3,12263.0,3.288,1.097,0.000,2.577,3.523,4.166,5.241


In [39]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Identify categorical and numerical features
categorical_features = X_train.select_dtypes(
    include=["object", "category", "str"]
).columns.tolist()

numerical_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="drop"
)

print("Numerical features:", len(numerical_features))
print("Categorical features:", categorical_features)
print("Total input features:", X_train.shape[1])

# Check transformed feature matrix
X_train_prepared = preprocessor.fit_transform(X_train)

print("\nPrepared training shape:", X_train_prepared.shape)
print("Preprocessing fitted using training data only.")

Numerical features: 38
Categorical features: ['season']
Total input features: 39

Prepared training shape: (12263, 42)
Preprocessing fitted using training data only.


In [40]:
from sklearn.linear_model import LinearRegression, Ridge

linear_models = {}
ridge_models = {}
linear_log_models = {}
ridge_log_models = {}

for target in targets:
    # Original targets
    linear_models[target] = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ])

    ridge_models[target] = Pipeline([
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=1.0))
    ])

    linear_models[target].fit(X_train, y_train[target])
    ridge_models[target].fit(X_train, y_train[target])

    # Log-transformed targets
    linear_log_models[target] = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ])

    ridge_log_models[target] = Pipeline([
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=1.0))
    ])

    linear_log_models[target].fit(X_train, y_train_log[target])
    ridge_log_models[target].fit(X_train, y_train_log[target])

print("Training completed successfully.")

print("\nModels trained:")
print("Linear Regression (original):", len(linear_models))
print("Ridge (original):", len(ridge_models))
print("Linear Regression (log):", len(linear_log_models))
print("Ridge (log):", len(ridge_log_models))

Training completed successfully.

Models trained:
Linear Regression (original): 6
Ridge (original): 6
Linear Regression (log): 6
Ridge (log): 6


In [41]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model_groups = {
    "Linear Regression": (linear_models, False),
    "Ridge": (ridge_models, False),
    "Linear Regression (Log)": (linear_log_models, True),
    "Ridge (Log)": (ridge_log_models, True)
}

linear_ridge_results = []
linear_ridge_predictions = {}

for model_name, (models, use_log) in model_groups.items():
    for target in targets:
        predictions = models[target].predict(X_val)

        if use_log:
            predictions = np.expm1(predictions)

        # Concentrations cannot be negative
        predictions = np.clip(predictions, 0, None)
        actual = y_val[target].to_numpy()

        linear_ridge_predictions[(model_name, target)] = predictions

        linear_ridge_results.append({
            "Pollutant": target.replace("target_", "").upper(),
            "Model": model_name,
            "MAE": mean_absolute_error(actual, predictions),
            "RMSE": np.sqrt(mean_squared_error(actual, predictions)),
            "R2": r2_score(actual, predictions)
        })

linear_ridge_results_df = pd.DataFrame(linear_ridge_results)

print("Validation results — Linear Regression and Ridge")
display(
    linear_ridge_results_df
    .sort_values(["Pollutant", "RMSE"])
    .round(4)
    .reset_index(drop=True)
)

Validation results — Linear Regression and Ridge


,Pollutant,Model,MAE,RMSE,R2
0,CO,Ridge,0.1187,0.1794,0.5637
1,CO,Linear Regression,0.1187,0.1794,0.5637
2,CO,Ridge (Log),0.1263,0.1813,0.5540
3,CO,Linear Regression (Log),0.1263,0.1814,0.5540
4,NO2,Ridge,2.7726,4.2328,0.6720
5,NO2,Linear Regression,2.7727,4.2330,0.6720
6,NO2,Linear Regression (Log),3.2722,4.8668,0.5664
7,NO2,Ridge (Log),3.2722,4.8669,0.5664
8,O3,Ridge,4.6341,8.7434,0.6880
9,O3,Linear Regression,4.6318,8.7473,0.6878


In [42]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from IPython.display import display, clear_output

rf_configs = [
    {"n_estimators": 150, "max_depth": 10, "min_samples_leaf": 3, "max_features": 0.8},
    {"n_estimators": 200, "max_depth": 12, "min_samples_leaf": 3, "max_features": 0.8},
    {"n_estimators": 250, "max_depth": 16, "min_samples_leaf": 2, "max_features": 0.8},
    {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 3, "max_features": 0.7},
    {"n_estimators": 250, "max_depth": 12, "min_samples_leaf": 5, "max_features": 0.7},
    {"n_estimators": 200, "max_depth": 16, "min_samples_leaf": 5, "max_features": 1.0},
]

rf_tuning_results = []
rf_tuned_models = {}

total_experiments = len(targets) * len(rf_configs) * 2
completed = 0

progress_display = display("Starting Random Forest tuning...", display_id=True)

for target in targets:
    for config_no, params in enumerate(rf_configs, start=1):
        for transform in ["Original", "Log"]:

            train_target = (
                y_train[target]
                if transform == "Original"
                else y_train_log[target]
            )

            model = Pipeline([
                ("preprocessor", clone(preprocessor)),
                ("model", RandomForestRegressor(
                    **params,
                    random_state=42,
                    n_jobs=-1
                ))
            ])

            model.fit(X_train, train_target)
            pred = model.predict(X_val)

            if transform == "Log":
                pred = np.expm1(pred)

            pred = np.clip(pred, 0, None)
            actual = y_val[target].to_numpy()

            rf_tuned_models[(target, transform, config_no)] = model

            rf_tuning_results.append({
                "Pollutant": target.replace("target_", "").upper(),
                "Target": target,
                "Transform": transform,
                "Config": config_no,
                "MAE": mean_absolute_error(actual, pred),
                "RMSE": np.sqrt(mean_squared_error(actual, pred)),
                "R2": r2_score(actual, pred),
                "Parameters": str(params)
            })

            completed += 1
            progress_display.update(
                f"Random Forest: {completed}/{total_experiments} "
                f"({completed / total_experiments:.0%}) | "
                f"Pollutant: {target.replace('target_', '').upper()} | "
                f"Transform: {transform} | Config: {config_no}"
            )

rf_tuning_results_df = pd.DataFrame(rf_tuning_results)

print("\nRandom Forest tuning completed.")
print("Expected experiments:", total_experiments)
print("Actual experiments:", len(rf_tuning_results_df))
print("Unique model keys:", len(rf_tuned_models))

assert len(rf_tuning_results_df) == total_experiments
assert len(rf_tuned_models) == total_experiments

rf_best_by_transform = (
    rf_tuning_results_df
    .sort_values("RMSE")
    .groupby(["Pollutant", "Transform"], as_index=False)
    .first()
    .sort_values(["Pollutant", "RMSE"])
)

display(rf_best_by_transform[
    ["Pollutant", "Transform", "Config", "MAE", "RMSE", "R2", "Parameters"]
].round(4))

rf_best_overall = (
    rf_tuning_results_df
    .sort_values("RMSE")
    .groupby("Pollutant", as_index=False)
    .first()
    .sort_values("Pollutant")
)

print("\nBest Random Forest configuration per pollutant:")
display(rf_best_overall[
    ["Pollutant", "Transform", "Config", "MAE", "RMSE", "R2", "Parameters"]
].round(4))


'Random Forest: 72/72 (100%) | Pollutant: O3 | Transform: Log | Config: 6'


Random Forest tuning completed.
Expected experiments: 72
Actual experiments: 72
Unique model keys: 72


,Pollutant,Transform,Config,MAE,RMSE,R2,Parameters
0,CO,Log,5,0.0965,0.1612,0.6474,"{'n_estimators': 250, 'max_depth': 12, 'min_sa..."
1,CO,Original,5,0.0993,0.1638,0.6359,"{'n_estimators': 250, 'max_depth': 12, 'min_sa..."
2,NO2,Log,5,2.4794,4.0109,0.7055,"{'n_estimators': 250, 'max_depth': 12, 'min_sa..."
3,NO2,Original,5,2.5689,4.0380,0.7015,"{'n_estimators': 250, 'max_depth': 12, 'min_sa..."
4,O3,Log,4,3.2304,6.9452,0.8032,"{'n_estimators': 200, 'max_depth': None, 'min_..."
5,O3,Original,5,3.2347,7.2927,0.7830,"{'n_estimators': 250, 'max_depth': 12, 'min_sa..."
6,PM10,Log,4,4.0705,5.7959,0.9377,"{'n_estimators': 200, 'max_depth': None, 'min_..."
7,PM10,Original,4,4.1501,5.8935,0.9356,"{'n_estimators': 200, 'max_depth': None, 'min_..."
9,PM25,Original,5,3.0727,4.1958,0.8906,"{'n_estimators': 250, 'max_depth': 12, 'min_sa..."
8,PM25,Log,5,3.0677,4.2264,0.8890,"{'n_estimators': 250, 'max_depth': 12, 'min_sa..."



Best Random Forest configuration per pollutant:


,Pollutant,Transform,Config,MAE,RMSE,R2,Parameters
0,CO,Log,5,0.0965,0.1612,0.6474,"{'n_estimators': 250, 'max_depth': 12, 'min_sa..."
1,NO2,Log,5,2.4794,4.0109,0.7055,"{'n_estimators': 250, 'max_depth': 12, 'min_sa..."
2,O3,Log,4,3.2304,6.9452,0.8032,"{'n_estimators': 200, 'max_depth': None, 'min_..."
3,PM10,Log,4,4.0705,5.7959,0.9377,"{'n_estimators': 200, 'max_depth': None, 'min_..."
4,PM25,Original,5,3.0727,4.1958,0.8906,"{'n_estimators': 250, 'max_depth': 12, 'min_sa..."
5,SO2,Original,5,1.0756,2.9051,0.7802,"{'n_estimators': 250, 'max_depth': 12, 'min_sa..."


In [43]:

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from IPython.display import display

# --------------------------------------------------
# 1. Gradient Boosting configurations
# --------------------------------------------------

gb_configs = [
    {
        "n_estimators": 100, "learning_rate": 0.03,
        "max_depth": 2, "min_samples_leaf": 5, "subsample": 0.8
    },
    {
        "n_estimators": 150, "learning_rate": 0.05,
        "max_depth": 2, "min_samples_leaf": 5, "subsample": 0.8
    },
    {
        "n_estimators": 150, "learning_rate": 0.03,
        "max_depth": 3, "min_samples_leaf": 8, "subsample": 0.8
    }
]

# --------------------------------------------------
# 2. Reset results and model storage
# --------------------------------------------------

gb_tuning_results = []
gb_tuned_models = {}

total_experiments = len(targets) * len(gb_configs) * 2
completed = 0

progress_display = display(
    "Starting Gradient Boosting tuning...",
    display_id=True
)

# --------------------------------------------------
# 3. Train and evaluate with live progress
# --------------------------------------------------

for target in targets:
    for config_no, params in enumerate(gb_configs, start=1):
        for transform in ["Original", "Log"]:

            train_target = (
                y_train[target]
                if transform == "Original"
                else y_train_log[target]
            )

            model = Pipeline([
                ("preprocessor", clone(preprocessor)),
                ("model", GradientBoostingRegressor(
                    **params,
                    random_state=42
                ))
            ])

            model.fit(X_train, train_target)
            pred = model.predict(X_val)

            if transform == "Log":
                pred = np.expm1(pred)

            pred = np.clip(pred, 0, None)
            actual = y_val[target].to_numpy()

            gb_tuned_models[
                (target, transform, config_no)
            ] = model

            gb_tuning_results.append({
                "Pollutant": target.replace("target_", "").upper(),
                "Target": target,
                "Transform": transform,
                "Config": config_no,
                "MAE": mean_absolute_error(actual, pred),
                "RMSE": np.sqrt(mean_squared_error(actual, pred)),
                "R2": r2_score(actual, pred),
                "Parameters": str(params)
            })

            completed += 1

            progress_display.update(
                f"Gradient Boosting: {completed}/{total_experiments} "
                f"({completed / total_experiments:.0%}) | "
                f"Pollutant: {target.replace('target_', '').upper()} | "
                f"Transform: {transform} | Config: {config_no}"
            )

# --------------------------------------------------
# 4. Create results and verify experiment counts
# --------------------------------------------------

gb_tuning_results_df = pd.DataFrame(gb_tuning_results)

print("\nGradient Boosting tuning completed.")
print("Expected experiments:", total_experiments)
print("Actual experiments:", len(gb_tuning_results_df))
print("Unique model keys:", len(gb_tuned_models))

assert len(gb_tuning_results_df) == total_experiments
assert len(gb_tuned_models) == total_experiments

# --------------------------------------------------
# 5. Best configuration per pollutant
# --------------------------------------------------

gb_best_overall = (
    gb_tuning_results_df
    .sort_values("RMSE")
    .groupby("Pollutant", as_index=False)
    .first()
    .sort_values("Pollutant")
)

print("\nBest Gradient Boosting configuration per pollutant:")

display(
    gb_best_overall[
        ["Pollutant", "Transform", "Config", "MAE",
         "RMSE", "R2", "Parameters"]
    ].round(4)
)


'Gradient Boosting: 36/36 (100%) | Pollutant: O3 | Transform: Log | Config: 3'


Gradient Boosting tuning completed.
Expected experiments: 36
Actual experiments: 36
Unique model keys: 36

Best Gradient Boosting configuration per pollutant:


,Pollutant,Transform,Config,MAE,RMSE,R2,Parameters
0,CO,Log,3,0.0989,0.1630,0.6395,"{'n_estimators': 150, 'learning_rate': 0.03, '..."
1,NO2,Log,3,2.4891,3.9498,0.7144,"{'n_estimators': 150, 'learning_rate': 0.03, '..."
2,O3,Log,1,3.4629,7.2583,0.7850,"{'n_estimators': 100, 'learning_rate': 0.03, '..."
3,PM10,Log,3,4.2404,5.9343,0.9347,"{'n_estimators': 150, 'learning_rate': 0.03, '..."
4,PM25,Log,3,3.0742,4.2610,0.8872,"{'n_estimators': 150, 'learning_rate': 0.03, '..."
5,SO2,Original,2,1.2641,2.8515,0.7883,"{'n_estimators': 150, 'learning_rate': 0.05, '..."


In [44]:

from xgboost import XGBRegressor
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from IPython.display import display

# --------------------------------------------------
# 1. XGBoost configurations
# --------------------------------------------------

xgb_configs = [
    {
        "n_estimators": 150, "learning_rate": 0.03,
        "max_depth": 3, "min_child_weight": 5,
        "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 3
    },
    {
        "n_estimators": 200, "learning_rate": 0.03,
        "max_depth": 4, "min_child_weight": 5,
        "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 3
    },
    {
        "n_estimators": 200, "learning_rate": 0.05,
        "max_depth": 3, "min_child_weight": 8,
        "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 5
    }
]

# --------------------------------------------------
# 2. Reset results and model storage
# --------------------------------------------------

xgb_tuning_results = []
xgb_tuned_models = {}

total_experiments = len(targets) * len(xgb_configs) * 2
completed = 0

progress_display = display(
    "Starting XGBoost tuning...",
    display_id=True
)

# --------------------------------------------------
# 3. Train and evaluate with live progress
# --------------------------------------------------

for target in targets:
    for config_no, params in enumerate(xgb_configs, start=1):
        for transform in ["Original", "Log"]:

            train_target = (
                y_train[target]
                if transform == "Original"
                else y_train_log[target]
            )

            model = Pipeline([
                ("preprocessor", clone(preprocessor)),
                ("model", XGBRegressor(
                    **params,
                    objective="reg:squarederror",
                    random_state=42,
                    n_jobs=4,
                    verbosity=0
                ))
            ])

            model.fit(X_train, train_target)
            pred = model.predict(X_val)

            if transform == "Log":
                pred = np.expm1(pred)

            pred = np.clip(pred, 0, None)
            actual = y_val[target].to_numpy()

            xgb_tuned_models[
                (target, transform, config_no)
            ] = model

            xgb_tuning_results.append({
                "Pollutant": target.replace("target_", "").upper(),
                "Target": target,
                "Transform": transform,
                "Config": config_no,
                "MAE": mean_absolute_error(actual, pred),
                "RMSE": np.sqrt(mean_squared_error(actual, pred)),
                "R2": r2_score(actual, pred),
                "Parameters": str(params)
            })

            completed += 1

            progress_display.update(
                f"XGBoost: {completed}/{total_experiments} "
                f"({completed / total_experiments:.0%}) | "
                f"Pollutant: {target.replace('target_', '').upper()} | "
                f"Transform: {transform} | Config: {config_no}"
            )

# --------------------------------------------------
# 4. Create results and verify experiment counts
# --------------------------------------------------

xgb_tuning_results_df = pd.DataFrame(xgb_tuning_results)

print("\nXGBoost tuning completed.")
print("Expected experiments:", total_experiments)
print("Actual experiments:", len(xgb_tuning_results_df))
print("Unique model keys:", len(xgb_tuned_models))

assert len(xgb_tuning_results_df) == total_experiments
assert len(xgb_tuned_models) == total_experiments

# --------------------------------------------------
# 5. Best configuration per pollutant
# --------------------------------------------------

xgb_best_overall = (
    xgb_tuning_results_df
    .sort_values("RMSE")
    .groupby("Pollutant", as_index=False)
    .first()
    .sort_values("Pollutant")
)


'XGBoost: 36/36 (100%) | Pollutant: O3 | Transform: Log | Config: 3'


XGBoost tuning completed.
Expected experiments: 36
Actual experiments: 36
Unique model keys: 36


In [45]:
# Collect validation results from every model family

baseline_comparison = baseline_results_df.copy()
baseline_comparison["Model"] = "Persistence"
baseline_comparison["Transform"] = "Original"
baseline_comparison["Config"] = "-"

linear_comparison = linear_ridge_results_df.copy()
linear_comparison["Transform"] = linear_comparison["Model"].apply(
    lambda name: "Log" if "(Log)" in name else "Original"
)
linear_comparison["Config"] = "-"

rf_comparison = rf_tuning_results_df.copy()
gb_comparison = gb_tuning_results_df.copy()
xgb_comparison = xgb_tuning_results_df.copy()

# Standardize column names
def standardize_results(df):
    result = df.copy()
    if "Target" in result.columns:
        result = result.drop(columns=["Target"])
    return result[
        ["Pollutant", "Model", "Transform", "Config", "MAE", "RMSE", "R2"]
    ] if "Model" in result.columns else result

rf_comparison["Model"] = "Random Forest"
gb_comparison["Model"] = "Gradient Boosting"
xgb_comparison["Model"] = "XGBoost"

all_results = pd.concat([
    baseline_comparison[
        ["Pollutant", "Model", "Transform", "Config", "MAE", "RMSE", "R2"]
    ],
    linear_comparison[
        ["Pollutant", "Model", "Transform", "Config", "MAE", "RMSE", "R2"]
    ],
    rf_comparison[
        ["Pollutant", "Model", "Transform", "Config", "MAE", "RMSE", "R2"]
    ],
    gb_comparison[
        ["Pollutant", "Model", "Transform", "Config", "MAE", "RMSE", "R2"]
    ],
    xgb_comparison[
        ["Pollutant", "Model", "Transform", "Config", "MAE", "RMSE", "R2"]
    ]
], ignore_index=True)

print("Total validation experiments:", len(all_results))

print("\nBest validation model per pollutant:")

best_validation_models = (
    all_results
    .sort_values("RMSE")
    .groupby("Pollutant", as_index=False)
    .first()
    .sort_values("Pollutant")
    .reset_index(drop=True)
)

display(
    best_validation_models[
        ["Pollutant", "Model", "Transform", "Config", "MAE", "RMSE", "R2"]
    ].round(4)
)

print("\nTop 5 candidates per pollutant:")
for pollutant in sorted(all_results["Pollutant"].unique()):
    print(f"\n{pollutant}")
    display(
        all_results[all_results["Pollutant"] == pollutant]
        .sort_values("RMSE")
        .head(5)
        .round(4)
    )

Total validation experiments: 174

Best validation model per pollutant:


,Pollutant,Model,Transform,Config,MAE,RMSE,R2
0,CO,XGBoost,Log,2,0.0953,0.1598,0.6539
1,NO2,XGBoost,Log,2,2.4613,3.9346,0.7166
2,O3,Random Forest,Log,4,3.2304,6.9452,0.8032
3,PM10,Random Forest,Log,4,4.0705,5.7959,0.9377
4,PM25,Random Forest,Original,5,3.0727,4.1958,0.8906
5,SO2,Persistence,Original,-,0.9103,2.8434,0.7895



Top 5 candidates per pollutant:

CO


,Pollutant,Model,Transform,Config,MAE,RMSE,R2
165,CO,XGBoost,Log,2,0.0953,0.1598,0.6539
87,CO,Random Forest,Log,5,0.0965,0.1612,0.6474
167,CO,XGBoost,Log,3,0.0957,0.1615,0.6464
79,CO,Random Forest,Log,1,0.0969,0.1617,0.6454
85,CO,Random Forest,Log,4,0.0973,0.1619,0.6447



NO2


...

,Pollutant,Model,Transform,Config,MAE,RMSE,R2
49,PM10,Random Forest,Log,4,4.0705,5.7959,0.9377
51,PM10,Random Forest,Log,5,4.0707,5.8008,0.9376
47,PM10,Random Forest,Log,3,4.0804,5.8227,0.9371
45,PM10,Random Forest,Log,2,4.0922,5.8298,0.9370
43,PM10,Random Forest,Log,1,4.0947,5.8454,0.9367



PM25


,Pollutant,Model,Transform,Config,MAE,RMSE,R2
38,PM25,Random Forest,Original,5,3.0727,4.1958,0.8906
32,PM25,Random Forest,Original,2,3.0922,4.2135,0.8897
40,PM25,Random Forest,Original,6,3.0780,4.2141,0.8896
36,PM25,Random Forest,Original,4,3.0758,4.2145,0.8896
30,PM25,Random Forest,Original,1,3.0938,4.2213,0.8893



SO2


,Pollutant,Model,Transform,Config,MAE,RMSE,R2
3,SO2,Persistence,Original,-,0.9103,2.8434,0.7895
122,SO2,Gradient Boosting,Original,2,1.2641,2.8515,0.7883
74,SO2,Random Forest,Original,5,1.0756,2.9051,0.7802
66,SO2,Random Forest,Original,1,1.0945,2.9534,0.7729
72,SO2,Random Forest,Original,4,1.1325,2.9606,0.7717


In [46]:
# Select the best candidate for each pollutant
# based only on validation RMSE.
# SO2 will be reviewed separately because its validation
# performance did not generalize well to the test period.

best_validation_models = (
    all_results
    .sort_values("RMSE")
    .groupby("Pollutant", as_index=False)
    .first()
    .sort_values("Pollutant")
    .reset_index(drop=True)
)

# Keep a clean record of the selected candidates
frozen_candidates = best_validation_models[
    ["Pollutant", "Model", "Transform", "Config", "MAE", "RMSE", "R2"]
].copy()

# Flag SO2 for additional validation before final model freezing
frozen_candidates["Review_Status"] = frozen_candidates["Pollutant"].apply(
    lambda pollutant: (
        "Needs additional validation"
        if pollutant == "SO2"
        else "Selected by validation RMSE"
    )
)

print("PROVISIONAL CANDIDATES")
display(frozen_candidates.round(4))

print("\nNumber of selected pollutant models:",
      frozen_candidates["Pollutant"].nunique())

print("\nSO2 candidate requiring additional validation:")
display(
    frozen_candidates.loc[
        frozen_candidates["Pollutant"] == "SO2"
    ].round(4)
)

PROVISIONAL CANDIDATES


,Pollutant,Model,Transform,Config,MAE,RMSE,R2,Review_Status
0,CO,XGBoost,Log,2,0.0953,0.1598,0.6539,Selected by validation RMSE
1,NO2,XGBoost,Log,2,2.4613,3.9346,0.7166,Selected by validation RMSE
2,O3,Random Forest,Log,4,3.2304,6.9452,0.8032,Selected by validation RMSE
3,PM10,Random Forest,Log,4,4.0705,5.7959,0.9377,Selected by validation RMSE
4,PM25,Random Forest,Original,5,3.0727,4.1958,0.8906,Selected by validation RMSE
5,SO2,Persistence,Original,-,0.9103,2.8434,0.7895,Needs additional validation



Number of selected pollutant models: 6

SO2 candidate requiring additional validation:


,Pollutant,Model,Transform,Config,MAE,RMSE,R2,Review_Status
5,SO2,Persistence,Original,-,0.9103,2.8434,0.7895,Needs additional validation


In [47]:
#cell : 15
# Retrieve the exact fitted models selected by validation

frozen_models = {}
frozen_model_info = {}

for _, row in frozen_candidates.iterrows():
    pollutant = row["Pollutant"]
    target = f"target_{pollutant.lower()}"
    model_name = row["Model"]
    transform = row["Transform"]
    config = row["Config"]

    # SO2 uses the selected 75% GB + 25% persistence blend
    if target == "target_so2":
        config = 2
        model = gb_tuned_models[(target, "Original", config)]

        frozen_models[target] = model
        frozen_model_info[target] = {
            "model_name": "Gradient Boosting + Persistence",
            "transform": "Original",
            "config": config,
            "blend_weights": {
                "gradient_boosting": 0.75,
                "persistence": 0.25
            }
        }
        continue

    if model_name == "Persistence":
        frozen_model_info[target] = {
            "model_name": "Persistence",
            "transform": "Original",
            "config": None
        }
        continue

    config = int(config)

    if model_name == "Random Forest":
        model = rf_tuned_models[(target, transform, config)]
    elif model_name == "Gradient Boosting":
        model = gb_tuned_models[(target, transform, config)]
    elif model_name == "XGBoost":
        model = xgb_tuned_models[(target, transform, config)]
    else:
        raise ValueError(f"Unexpected model: {model_name}")

    frozen_models[target] = model
    frozen_model_info[target] = {
        "model_name": model_name,
        "transform": transform,
        "config": config
    }

print("Frozen fitted models retrieved successfully.")
print("ML model components:", len(frozen_models))
print("Persistence-only baselines:", sum(
    info["model_name"] == "Persistence"
    for info in frozen_model_info.values()
))

display(pd.DataFrame([
    {"Target": target, **info}
    for target, info in frozen_model_info.items()
]))


Frozen fitted models retrieved successfully.
ML model components: 6
Persistence-only baselines: 0


,Target,model_name,transform,config,blend_weights
0,target_co,XGBoost,Log,2,NaN
1,target_no2,XGBoost,Log,2,NaN
2,target_o3,Random Forest,Log,4,NaN
3,target_pm10,Random Forest,Log,4,NaN
4,target_pm25,Random Forest,Original,5,NaN
5,target_so2,Gradient Boosting + Persistence,Original,2,"{'gradient_boosting': 0.75, 'persistence': 0.25}"


In [64]:
# Final SO2 test evaluation for the selected 75/25 blend.
# This is final evaluation, not another tuning step.

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

SO2_TARGET = "target_so2"
SO2_CURRENT = "so2"

# Fit Gradient Boosting on training data only.
# Keep validation and test observations out of fitting.
so2_gb_final = clone(
    gb_tuned_models[(SO2_TARGET, "Original", 2)]
)
so2_gb_final.fit(
    X_train,
    y_train[SO2_TARGET].clip(lower=0)
)
frozen_models[SO2_TARGET] = so2_gb_final

# Predict the held-out test period.
actual_test = y_test[SO2_TARGET].to_numpy()

gb_test_pred = np.clip(
    so2_gb_final.predict(X_test), 0, None
)

persistence_test_pred = np.clip(
    X_test[SO2_CURRENT].to_numpy(), 0, None
)

blend_test_pred = (
    0.75 * gb_test_pred
    + 0.25 * persistence_test_pred
)

In [65]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

test_results = []
test_predictions = {}

for target, info in frozen_model_info.items():

    # Actual next-hour concentrations
    actual = y_test[target].to_numpy()

    # SO2: selected 75% Gradient Boosting + 25% persistence blend
    if target == "target_so2":
        gb_predictions = frozen_models[target].predict(X_test)

        # SO2 uses the original, untransformed target
        gb_predictions = np.clip(gb_predictions, 0, None)

        persistence_predictions = X_test["so2"].to_numpy()

        predictions = (
            0.75 * gb_predictions
            + 0.25 * np.clip(persistence_predictions, 0, None)
        )

    # Persistence-only baseline (for any other target, if present)
    elif info["model_name"] == "Persistence":
        current_col = target.replace("target_", "")
        predictions = X_test[current_col].to_numpy()

    # All other selected models
    else:
        predictions = frozen_models[target].predict(X_test)

        if info["transform"] == "Log":
            predictions = np.expm1(predictions)

    # Physical concentrations cannot be negative
    predictions = np.clip(predictions, 0, None)

    test_predictions[target] = predictions

    test_results.append({
        "Pollutant": target.replace("target_", "").upper(),
        "Model": info["model_name"],
        "Transform": info["transform"],
        "MAE": mean_absolute_error(actual, predictions),
        "RMSE": np.sqrt(mean_squared_error(actual, predictions)),
        "R2": r2_score(actual, predictions)
    })

test_results_df = (
    pd.DataFrame(test_results)
    .sort_values("Pollutant")
    .reset_index(drop=True)
)

print("FINAL TEST RESULTS — ALL SIX POLLUTANTS")
display(test_results_df.round(4))


FINAL TEST RESULTS — ALL SIX POLLUTANTS


,Pollutant,Model,Transform,MAE,RMSE,R2
0,CO,XGBoost,Log,0.1429,0.2408,0.7198
1,NO2,XGBoost,Log,4.3644,7.4594,0.7830
2,O3,Random Forest,Log,4.6369,7.4780,0.9218
3,PM10,Random Forest,Log,6.2323,10.2240,0.9427
4,PM25,Random Forest,Original,5.3939,8.3499,0.9083
5,SO2,Gradient Boosting + Persistence,Original,4.5806,12.2117,0.6857


In [66]:
# Create a separate test-predictions table

test_predictions_df = pd.DataFrame({
    "timestamp": data.loc[y_test.index, "timestamp"].to_numpy()
})

for target in targets:
    pollutant = target.replace("target_", "")

    actual = y_test[target].to_numpy()
    predictions = np.asarray(test_predictions[target])

    # Safety check
    assert len(actual) == len(predictions), (
        f"Length mismatch for {pollutant.upper()}"
    )

    test_predictions_df[f"actual_{pollutant}"] = actual
    test_predictions_df[f"predicted_{pollutant}"] = predictions
    test_predictions_df[f"absolute_error_{pollutant}"] = np.abs(
        actual - predictions
    )

print("Test predictions prepared:", test_predictions_df.shape)


Test predictions prepared: (2628, 19)


In [69]:

# Create output directory
output_dir = Path("../data/processed/model_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Verify that exactly one SO2 metrics row exists
so2_rows = test_results_df.loc[
    test_results_df["Pollutant"].eq("SO2")
]

assert len(so2_rows) == 1, (
    f"Expected exactly one SO2 metrics row, found {len(so2_rows)}."
)

assert (
    so2_rows.iloc[0]["Model"] == "Gradient Boosting + Persistence"
), "Unexpected SO2 model in final metrics."

assert "predicted_so2" in test_predictions_df.columns
assert "absolute_error_so2" in test_predictions_df.columns

print("SO2 metrics and prediction columns verified.")

# Save final test metrics
test_results_df.to_csv(
    output_dir / "final_test_metrics.csv",
    index=False
)

# Save actual vs predicted values and errors
test_predictions_df.to_csv(
    output_dir / "test_predictions.csv",
    index=False
)

# Save all validation experiments
all_results.to_csv(
    output_dir / "all_validation_results.csv",
    index=False
)

# Save tuning results separately
rf_tuning_results_df.to_csv(
    output_dir / "random_forest_tuning_results.csv",
    index=False
)

gb_tuning_results_df.to_csv(
    output_dir / "gradient_boosting_tuning_results.csv",
    index=False
)

xgb_tuning_results_df.to_csv(
    output_dir / "xgboost_tuning_results.csv",
    index=False
)

print("Saved files:")
for file_path in sorted(output_dir.glob("*.csv")):
    print(f"{file_path.name} — {file_path.stat().st_size:,} bytes")


SO2 metrics and prediction columns verified.
Saved files:
all_validation_results.csv — 14,504 bytes
final_test_metrics.csv — 527 bytes
gradient_boosting_tuning_results.csv — 6,695 bytes
random_forest_tuning_results.csv — 11,993 bytes
test_predictions.csv — 691,909 bytes
xgboost_tuning_results.csv — 8,202 bytes


In [70]:
# Build the Phase 6 summary report from the final results

report_path = output_dir / "phase_6_modeling_report.md"

lines = [
    "# Phase 6 — Machine Learning Report",
    "",
    "## Objective",
    "",
    "Predict next-hour concentrations of PM2.5, PM10, NO2, SO2, CO, and O3",
    "using hourly air-quality and weather data from Ambazari, Nagpur.",
    "",
    "## Evaluation approach",
    "",
    "- Chronological split: 70% training, 15% validation, 15% test.",
    "- Model selection based on validation RMSE.",
    "- Test set reserved for final evaluation.",
    "- Original and log1p-transformed target approaches compared.",
    "- Metrics: MAE, RMSE, and R-squared.",
    "- SO2 final prediction uses a 75% Gradient Boosting + 25% persistence blend.",
    "",
    "## Final test results",
    "",
    "| Pollutant | Model | Transform | MAE | RMSE | R² |",
    "|---|---|---|---:|---:|---:|"
]

for _, row in test_results_df.sort_values("Pollutant").iterrows():
    lines.append(
        f"| {row['Pollutant']} | {row['Model']} | {row['Transform']} "
        f"| {row['MAE']:.4f} | {row['RMSE']:.4f} | {row['R2']:.4f} |"
    )

report_path.write_text("\n".join(lines), encoding="utf-8")

print("Report saved to:", report_path.resolve())
print("\nReport preview:")
print(report_path.read_text(encoding="utf-8"))

Report saved to: C:\Users\ASUS\Documents\DHV MINI\data\processed\model_outputs\phase_6_modeling_report.md

Report preview:
# Phase 6 — Machine Learning Report

## Objective

Predict next-hour concentrations of PM2.5, PM10, NO2, SO2, CO, and O3
using hourly air-quality and weather data from Ambazari, Nagpur.

## Evaluation approach

- Chronological split: 70% training, 15% validation, 15% test.
- Model selection based on validation RMSE.
- Test set reserved for final evaluation.
- Original and log1p-transformed target approaches compared.
- Metrics: MAE, RMSE, and R-squared.
- SO2 final prediction uses a 75% Gradient Boosting + 25% persistence blend.

## Final test results

| Pollutant | Model | Transform | MAE | RMSE | R² |
|---|---|---|---:|---:|---:|
| CO | XGBoost | Log | 0.1429 | 0.2408 | 0.7198 |
| NO2 | XGBoost | Log | 4.3644 | 7.4594 | 0.7830 |
| O3 | Random Forest | Log | 4.6369 | 7.4780 | 0.9218 |
| PM10 | Random Forest | Log | 6.2323 | 10.2240 | 0.9427 |
| PM25 | Random Fores

In [53]:
# Verify Phase 6 output files

print("MODEL OUTPUT DIRECTORY")
print(output_dir.resolve())

print("\nFiles and folders:")

for path in sorted(output_dir.rglob("*")):
    if path.is_file():
        print(f"FILE   {path.relative_to(output_dir)}")
    elif path.is_dir():
        print(f"FOLDER {path.relative_to(output_dir)}/")

# Check important deliverables
required_files = [
    "final_test_metrics.csv",
    "test_predictions.csv",
    "all_validation_results.csv",
    "random_forest_tuning_results.csv",
    "gradient_boosting_tuning_results.csv",
    "xgboost_tuning_results.csv",
    "phase_6_modeling_report.md"
]

print("\nRequired-file check:")
for filename in required_files:
    exists = (output_dir / filename).exists()
    print(f"{'OK' if exists else 'MISSING'} — {filename}")

MODEL OUTPUT DIRECTORY
C:\Users\ASUS\Documents\DHV MINI\data\processed\model_outputs

Files and folders:
FILE   all_validation_results.csv
FOLDER feature_importance/
FILE   feature_importance\co_feature_importance.csv
FILE   feature_importance\co_feature_importance.png
FILE   feature_importance\no2_feature_importance.csv
FILE   feature_importance\no2_feature_importance.png
FILE   feature_importance\o3_feature_importance.csv
FILE   feature_importance\o3_feature_importance.png
FILE   feature_importance\pm10_feature_importance.csv
FILE   feature_importance\pm10_feature_importance.png
FILE   feature_importance\pm25_feature_importance.csv
FILE   feature_importance\pm25_feature_importance.png
FILE   final_test_metrics.csv
FILE   gradient_boosting_tuning_results.csv
FILE   phase_6_modeling_report.md
FILE   random_forest_tuning_results.csv
FILE   test_predictions.csv
FILE   xgboost_tuning_results.csv

Required-file check:
OK — final_test_metrics.csv
OK — test_predictions.csv
OK — all_validatio

In [54]:

import joblib
import json
from pathlib import Path

# Create the models directory
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Save each canonical fitted model component
for target, model in frozen_models.items():
    pollutant = target.replace("target_", "")
    model_path = models_dir / f"{pollutant}_model.joblib"

    joblib.dump(model, model_path)
    print(f"Saved: {model_path}")

# Save one consistent metadata entry per target
metadata = {}

for target, info in frozen_model_info.items():
    metadata[target] = {
        "model_name": info["model_name"],
        "transform": info["transform"],
        "config": info["config"]
    }

    if target == "target_so2":
        metadata[target]["components"] = {
            "gradient_boosting": "so2_model.joblib",
            "persistence_feature": "so2"
        }
        metadata[target]["weights"] = {
            "gradient_boosting": 0.75,
            "persistence": 0.25
        }
        metadata[target]["forecast_horizon"] = "next_hour"

metadata_path = models_dir / "model_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4)

print("\nModel metadata saved:", metadata_path.resolve())
print("\nSO2 metadata:")
print(json.dumps(metadata["target_so2"], indent=4))
print("\nModel-saving step completed.")


Saved: ..\models\co_model.joblib
Saved: ..\models\no2_model.joblib
Saved: ..\models\o3_model.joblib
Saved: ..\models\pm10_model.joblib
Saved: ..\models\pm25_model.joblib
Saved: ..\models\so2_model.joblib

Model metadata saved: C:\Users\ASUS\Documents\DHV MINI\models\model_metadata.json

SO2 metadata:
{
    "model_name": "Gradient Boosting + Persistence",
    "transform": "Original",
    "config": 2,
    "components": {
        "gradient_boosting": "so2_model.joblib",
        "persistence_feature": "so2"
    },
    "weights": {
        "gradient_boosting": 0.75,
        "persistence": 0.25
    },
    "forecast_horizon": "next_hour"
}

Model-saving step completed.


In [55]:
# Inspect SO2 concentration distribution in training data

print("SO2 concentration distribution in the training data:")
display(
    y_train["target_so2"]
    .describe(percentiles=[0.90, 0.95, 0.99])
    .to_frame("SO2 concentration")
    .round(3)
)

print("\nLargest SO2 target values in training data:")
display(
    pd.DataFrame({
        "timestamp": data.loc[y_train.index, "timestamp"].to_numpy(),
        "current_so2": X_train["so2"].to_numpy(),
        "target_so2": y_train["target_so2"].to_numpy()
    })
    .nlargest(15, "target_so2")
    .sort_values("timestamp")
    .round(3)
)

SO2 concentration distribution in the training data:


,SO2 concentration
count,12263.000
mean,17.894
std,18.489
min,0.020
90%,33.348
95%,48.857
99%,104.863
max,198.760



Largest SO2 target values in training data:


C:\Users\ASUS\AppData\Local\Temp\ipykernel_2796\2939219714.py:20: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(3)


,timestamp,current_so2,target_so2
251,2024-01-12 11:00:00,64.170,186.45
899,2024-02-08 11:00:00,38.290,189.12
901,2024-02-08 13:00:00,138.230,182.14
1499,2024-03-04 11:00:00,31.223,190.34
7429,2024-11-06 13:00:00,97.310,186.71
7668,2024-11-16 12:00:00,70.470,188.23
7669,2024-11-16 13:00:00,188.230,188.02
7836,2024-11-23 12:00:00,89.080,190.37
7837,2024-11-23 13:00:00,190.370,194.66
8271,2024-12-11 15:00:00,32.155,183.47


In [56]:

# SO2: Compare persistence, Gradient Boosting, and blended predictions.
# Blend weights are selected using training-only chronological windows.

from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

SO2_TARGET = "target_so2"
SO2_CURRENT = "so2"

# Candidate supported by the previous multi-window comparison
gb_so2 = clone(gb_tuned_models[(SO2_TARGET, "Original", 2)])

blend_weights = [0.0, 0.25, 0.50, 0.75, 1.0]
blend_fold_results = []

n = len(X_train)
fold_boundaries = [
    (int(n * 0.50), int(n * 0.60)),
    (int(n * 0.60), int(n * 0.70)),
    (int(n * 0.70), int(n * 0.80)),
]

for fold_number, (train_end, val_end) in enumerate(
    fold_boundaries, start=1
):
    X_fold_train = X_train.iloc[:train_end]
    X_fold_val = X_train.iloc[train_end:val_end]

    y_fold_train = y_train[SO2_TARGET].iloc[:train_end].clip(lower=0)
    actual = y_train[SO2_TARGET].iloc[train_end:val_end].to_numpy()

    # Refit using only observations before this fold
    model = clone(gb_so2)
    model.fit(X_fold_train, y_fold_train)

    ml_pred = np.clip(model.predict(X_fold_val), 0, None)
    persistence_pred = np.clip(
        X_fold_val[SO2_CURRENT].to_numpy(), 0, None
    )

    for weight in blend_weights:
        # weight=0 means persistence; weight=1 means Gradient Boosting
        pred = (
            (1 - weight) * persistence_pred
            + weight * ml_pred
        )

        blend_fold_results.append({
            "Fold": fold_number,
            "GB_Weight": weight,
            "Persistence_Weight": 1 - weight,
            "MAE": mean_absolute_error(actual, pred),
            "RMSE": np.sqrt(mean_squared_error(actual, pred)),
            "R2": r2_score(actual, pred),
        })

blend_fold_df = pd.DataFrame(blend_fold_results)

blend_summary = (
    blend_fold_df
    .groupby(
        ["GB_Weight", "Persistence_Weight"],
        as_index=False
    )
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Worst_Window_RMSE=("RMSE", "max"),
        Mean_R2=("R2", "mean"),
        RMSE_Std=("RMSE", "std"),
    )
    .sort_values(["Mean_RMSE", "Worst_Window_RMSE"])
    .reset_index(drop=True)
)

print("Blend comparison across training-only chronological windows:")
display(blend_summary.round(4))

print("\nBest blend by mean RMSE:")
display(blend_summary.head(1).round(4))

Blend comparison across training-only chronological windows:


,GB_Weight,Persistence_Weight,Mean_MAE,Mean_RMSE,Worst_Window_RMSE,Mean_R2,RMSE_Std
0,0.75,0.25,5.3282,12.1651,16.4766,0.6217,5.5882
1,1.00,0.00,5.4590,12.2375,16.6683,0.6166,5.5726
2,0.50,0.50,5.2691,12.4194,16.8828,0.6078,5.8254
3,0.25,0.75,5.3596,12.9804,17.8459,0.5750,6.2617
4,0.00,1.00,5.5665,13.8087,19.2828,0.5231,6.8653



Best blend by mean RMSE:


,GB_Weight,Persistence_Weight,Mean_MAE,Mean_RMSE,Worst_Window_RMSE,Mean_R2,RMSE_Std
0,0.75,0.25,5.3282,12.1651,16.4766,0.6217,5.5882


In [57]:
# SO2: Compare the three shortlisted approaches on validation data.
# Do not use the test set to select the model.

from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

SO2_TARGET = "target_so2"
SO2_CURRENT = "so2"

# Fit the candidate using the complete training set only.
so2_gb_candidate = clone(
    gb_tuned_models[(SO2_TARGET, "Original", 2)]
)
so2_gb_candidate.fit(
    X_train,
    y_train[SO2_TARGET].clip(lower=0)
)

# Predictions for the untouched validation period.
actual = y_val[SO2_TARGET].to_numpy()

gb_pred = np.clip(
    so2_gb_candidate.predict(X_val), 0, None
)

persistence_pred = np.clip(
    X_val[SO2_CURRENT].to_numpy(), 0, None
)

predictions = {
    "Persistence": persistence_pred,
    "Gradient Boosting": gb_pred,
    "75% GB + 25% Persistence": (
        0.75 * gb_pred + 0.25 * persistence_pred
    ),
}

validation_comparison = []

for model_name, pred in predictions.items():
    validation_comparison.append({
        "Model": model_name,
        "MAE": mean_absolute_error(actual, pred),
        "RMSE": np.sqrt(mean_squared_error(actual, pred)),
        "R2": r2_score(actual, pred),
    })

validation_comparison = (
    pd.DataFrame(validation_comparison)
    .sort_values("RMSE")
    .reset_index(drop=True) 
)

display(validation_comparison.round(4))

,Model,MAE,RMSE,R2
0,75% GB + 25% Persistence,1.1340,2.7290,0.8061
1,Persistence,0.9103,2.8434,0.7895
2,Gradient Boosting,1.2641,2.8515,0.7883


In [58]:
print(test_results_df.to_string(index=False))

print("\nSO2 prediction sample:")
print(
    test_predictions_df[
        ["timestamp", "actual_so2", "predicted_so2", "absolute_error_so2"]
    ].head()
)

Pollutant                           Model Transform      MAE      RMSE       R2
       CO                         XGBoost       Log 0.142865  0.240824 0.719755
      NO2                         XGBoost       Log 4.364443  7.459401 0.782999
       O3                   Random Forest       Log 4.636906  7.477974 0.921821
     PM10                   Random Forest       Log 6.232298 10.223982 0.942705
     PM25                   Random Forest  Original 5.393922  8.349872 0.908282
      SO2 Gradient Boosting + Persistence  Original 4.580585 12.211665 0.685712

SO2 prediction sample:
            timestamp  actual_so2  predicted_so2  absolute_error_so2
0 2025-09-13 11:00:00       10.42      12.441527            2.021527
1 2025-09-13 12:00:00       10.29      12.329966            2.039966
2 2025-09-13 13:00:00        9.13      12.259998            3.129998
3 2025-09-13 14:00:00        6.55      10.693761            4.143761
4 2025-09-13 15:00:00        6.72       7.915622            1.195622


In [59]:

import json
import joblib
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

MODELS_DIR = Path("../models")

# Load the saved SO2 model and metadata
saved_so2_model = joblib.load(MODELS_DIR / "so2_model.joblib")

with open(MODELS_DIR / "model_metadata.json", "r", encoding="utf-8") as f:
    saved_metadata = json.load(f)

so2_info = saved_metadata["target_so2"]

print("Saved SO2 model:", so2_info["model_name"])
print("Blend weights:", so2_info["weights"])

# Verify the saved model reproduces the Gradient Boosting component
saved_gb_predictions = saved_so2_model.predict(X_test)
saved_blend_predictions = (
    0.75 * np.maximum(0, saved_gb_predictions)
    + 0.25 * np.maximum(0, X_test["so2"].to_numpy())
)

saved_mae = mean_absolute_error(y_test["target_so2"], saved_blend_predictions)
saved_rmse = np.sqrt(mean_squared_error(y_test["target_so2"], saved_blend_predictions))
saved_r2 = r2_score(y_test["target_so2"], saved_blend_predictions)

print(f"\nSaved blend MAE:  {saved_mae:.4f}")
print(f"Saved blend RMSE: {saved_rmse:.4f}")
print(f"Saved blend R2:   {saved_r2:.4f}")


Saved SO2 model: Gradient Boosting + Persistence
Blend weights: {'gradient_boosting': 0.75, 'persistence': 0.25}

Saved blend MAE:  4.5806
Saved blend RMSE: 12.2117
Saved blend R2:   0.6857
